### 파이썬이 제공하는 다양한 종류의 자료구조를 살펴보자.

# 1. 시퀀스(Sequence)

- 순서(order) 가 있음
- 인덱싱 가능 (obj[i])
- 슬라이싱 가능 (obj[a:b:c])
- 길이 있음 (len())
- 반복(iteration) 가능 (for) 

 | 타입          | 변경 가능?      | 예시                  |
| ----------- | ----------- | ------------------- |
| `list`      | ⭕ mutable   | `[1, 2, 3]`         |
| `tuple`     | ❌ immutable | `(1, 2, 3)`         |
| `str`       | ❌ immutable | `"abc"`             |
| `range`     | ❌ immutable | `range(0, 10)`      |
| `bytes`     | ❌ immutable | `b"abc"`            |
| `bytearray` | ⭕ mutable   | `bytearray(b"abc")` |
 

In [3]:
s = [10, 20, 30]

print(s[0])      # 10
print(s[-1])     # 30
print(s[0:2])    # [10, 20]
print(len(s))    # 3

10
30
[10, 20]
3


## 1.1 Sequence 공통 연산

### 인덱싱

In [ ]:
s = [10, 20, 30, 40, 50]

print(s[0])  # 인덱싱. index로 요소를 조회
print(s[-1]) # 끝에서 첫번째

print(s[1:3]) # index 1~3, 마지막 인덱스는 포함x
print(s[::-1])   # 뒤집기


10
50
[20, 30]
[50, 40, 30, 20, 10]


### 결합과 반복

In [12]:
print([1, 2] + [3, 4])   # [1,2,3,4]
print("ab" * 3)          # "ababab"

[1, 2, 3, 4]
ababab


### Sequence Protocol - 덕 타이핑(duck typing)
- 시퀀스처럼 행동하면 시퀀스
- 최소 요구사항  
__ len__()  
__ getitem__()  

In [ ]:
class MySeq: # __getitem__, __len__을 가지고 있으니까 시퀀스로 간주됨.
    def __getitem__(self, index):
        return index * 10

    def __len__(self):
        return 5

s = MySeq()
print(s[2])       # 20
print(len(s))     # 5
print(isinstance(s, Sequence)) # False. isinstance()는 상속 여부를 확인

20
5
True


In [34]:
class MySeq: 
    def __getitem__(self, index):
        return index * 10

    # def __len__(self):
    #     return 5

# if isinstance(x, Sequence): # Sequence인지 확인할 때 instance()대신
# try-except로 확인
try:
    s = MySeq()
    s[0]   # __getitem__ 호출
    len(s) # __len__ 호출
except TypeError:
    print("this is not a sequence")

this is not a sequence


In [ ]:
from collections.abc import Sequence

print(isinstance([1,2,3], Sequence))   # True
print(isinstance("abc", Sequence))     # True
print(isinstance({1,2,3}, Sequence))   # False. Set은 순서가 없음.

class MySeq(Sequence): # Sequence를 명시적으로 상속
    def __getitem__(self, index):
        return index * 10

    def __len__(self):
        return 5
    
print(isinstance(s, Sequence)) # True

True
True
False
True


## 1.2 slice()
- 슬라이싱 문법의 실체인 객체(immutable)
- slice(start, stop, step)

In [49]:
s = slice(1, 5, 2)  
print(s)  # slice(1, 5, 2)
print(f"{s.start=}")
print(f"{s.stop=}")
print(f"{s.step=}")

slice(1, 5, 2)
s.start=1
s.stop=5
s.step=2


In [47]:
str = "0123456789"
print(str[1])     # str.__getitem__(slice(1, 2, 1))과 동일
print(str[1:6])   # str.__getitem__(slice(1, 6, 1))과 동일. 범위의 끝은 미포함
print(str[1:6:2]) # str.__getitem__(slice(1, 5, 2))와 동일

1
12345
135


## 1.3 iterable, iterator
- Iterable  
반복할 수 있는 객체  
iter(x) 가 되는 것  

- Iterator  
반복 중인 상태를 가진 객체. 일회용.  
next(x) 가 되는 것  
- iterable → iter() → iterator → next()

### Iterable의 조건
__ iter__() 가 있고, 그 결과가 iterator 여야 함

In [ ]:
class MyIterable:        # iterable은 __iter__를 가지고 있어야 한다.
    def __iter__(self):  # __iter__는 iterator를 반환해야 한다.
        return iter([1, 2, 3]) # iterator를 반환

# MyIterable객체의 __iter__를 자동 호출. 
for x in MyIterable():   # iterator의 __next__를 반복적으로 호출
    print(x)

1
2
3


### Iterator의 조건
Iterator는 자기 자신을 반환함.
- __ iter__() → return self
- __ next__() → 값 반환 or StopIteration

In [53]:
class Counter:  # iterable인 동시에 iterator
    def __init__(self, n):
        self.i = 0
        self.n = n

    def __iter__(self):    # iterable이 가져야하는 함수
        return self

    def __next__(self):    # iterator가 가져야하는 함수
        if self.i >= self.n:
            raise StopIteration
        val = self.i
        self.i += 1
        return val

for x in Counter(5):
    print(x)

0
1
2
3
4


### generator - iterator를 자동 구현( __ iter__ + __ next__)

In [65]:
def gen():
    yield 1   # next()가 호출되면 yield까지 실행하고 1을 반환
    yield 2   # next()가 호출되면 yield까지 실행하고 2를 반환

g = gen()

print(next(g))  # 1
print(next(g))  # 2
# print(next(g))  # StopIteration

g = gen() # 일회용이므로 iterator를 다시 얻어와야함
while True:
    try:
        x = next(g)
    except StopIteration:  # StopIteration은 에러가 아니라 "정상 종료 신호"
        break
     
for x in gen(): # 일회용이므로 iterator를 다시 얻어와야함
    print(x)

1
2
1
2


### 제너레이터는 slice불가. 별도의 모듈로 처리

In [211]:
from itertools import islice

def numbers():   # 제네레이터
    for i in range(10):
        yield i

gen = numbers()
result = islice(gen, 5) 
list(result)

[0, 1, 2, 3, 4]

## 제너레이터는 chain, + 연산 불가

In [212]:
from itertools import chain

def a():
    for i in range(3):
        yield i
        
gen = a()
it = chain(gen, [3, 4, 5])
list(it)

[0, 1, 2, 3, 4, 5]

### 제너레이터 (이터레이터)는 계속 반복 불가

In [219]:
from itertools import cycle

# def a():
#     for i in range(3):
#         yield i

def a() :
    yield from range(3)   # 위의 코드를 간단히 가능

gen = a() # 0, 1, 2

it = cycle(gen) # 0, 1, 2, 0, 1, 2, 0, 1, 2, ...    

for _ in range(6):
    print(next(it))    

0
1
2
0
1
2


## 1.4 Mutable Sequence와 Immutable Sequence

### Mutable Sequence

- list, bytearray
- 요소 변경 ⭕
- 해시 불가

In [218]:
lst = [1, 2, 3] # list는 가변(mutable)
lst[0] = 10 
print(lst[0])

10


### Immutable Sequence
- str, tuple, bytes, range
- 요소 변경 ❌
- 해시 가능 → dict key 가능

In [19]:
t = (1, 2, 3) # tuple은 불변(immutable)
print(t[0])
t[0] = 10     # 변경불가. TypeError: 'tuple' object does not support item assignment

1


TypeError: 'tuple' object does not support item assignment

# 2. list 
- 가변(mutable) · 순서 있음(MutableSequence) · 동적 참조 배열(dynamic array)
- 서로 다른 타입 혼합 가능
- Sequence는 읽기 전용, MutableSequence는 쓰기 연산 추가된 Sequence
- Iterable  
 &nbsp;&nbsp;&nbsp;&nbsp;└── Sequence  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└── MutableSequence  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;└── list  


## 2.1 list생성하기

In [97]:
# list생성방법
lst = []     # 빈 list
lst = list() # 빈 list
lst = [1,"abc",True] # 서로 다른 타입의 객체도 저장가능
print(lst)
lst = [*"abc"] # ['a', 'b', 'c'] # 언패킹(unpacking)
print(lst)
lst = [*range(5)] # [0, 1, 2, 3, 4]
print(lst)
lst = [*[1, 2], *[3, 4], 5] # [1, 2, 3, 4, 5]
print(lst)

[1, 'abc', True]
['a', 'b', 'c']
[0, 1, 2, 3, 4]
[1, 2, 3, 4, 5]


### list()로 생성하기

In [103]:
from itertools import repeat

print(list((1,2,3)))     # 튜플 -> list
print(list({1, 2, 3}))   # set -> list
print(list("abc"))       # ['a', 'b', 'c']
print(list(range(5)))    # [0, 1, 2, 3, 4]
print(list(repeat(0,5)))  # [0, 0, 0, 0, 0]
print(list(map(lambda x: x * 2, range(5)))) # [0, 2, 4, 6, 8]
print(list(filter(lambda x: x % 2 == 0, range(10)))) # [0, 2, 4, 6, 8]

[1, 2, 3]
[1, 2, 3]
['a', 'b', 'c']
[0, 1, 2, 3, 4]
[0, 0, 0, 0, 0]
[0, 2, 4, 6, 8]
[0, 2, 4, 6, 8]


### list comprehension으로 생성하기

In [89]:
# list생성방법
lst = [x for x in range(5)] # 0,1,2,3,4
print(lst)
lst = [x for x in range(5) if x%2==0] # 0,2,4
print(lst)
lst = [x*x for x in range(5) if x%2==0] # 0,4,16
print(lst)
lst = [(i, j) for i in range(3) for j in range(2)]
print(lst)

[0, 1, 2, 3, 4]
[0, 2, 4]
[0, 4, 16]
[(0, 0), (0, 1), (1, 0), (1, 1), (2, 0), (2, 1)]


## 2.2 list 변경 & 복사

In [83]:
lst = [1,2,3]

lst.append(4) # 마지막에 새로운 요소 추가
print(lst)
lst.insert(0, 9) # index 0위치에 9추가
print(lst)
lst = lst + [5,6]
print(lst)
lst.remove(9) # 9를 찾아서 삭제
print(lst)

[1, 2, 3, 4]
[9, 1, 2, 3, 4]
[9, 1, 2, 3, 4, 5, 6]
[1, 2, 3, 4, 5, 6]


### list의 복사

In [98]:
a = [1, 2, 3]
b = a.copy()  # 얕은 복사
b = a[:]      # 얕은 복사
b = list(a)

print(a)
print(b)

[1, 2, 3]
[1, 2, 3]


### list 슬라이싱 - 새 list 생성

In [69]:
a = [1, 2, 3, 4]
b = a[1:3]

b is a  # False

False

## 2.3 list의 메서드
- append
- extend
- insert
- remove
- pop
- clear
- index
- count
- sort
- reverse
- copy


In [112]:
lst = [3,1,2,2]
lst.sort()    # 정렬
print(lst)
lst.reverse() # 역순
print(lst)
print(lst.index(2)) # 2의 위치. 여러개일 경우 먼저 발견된 것의 위치
print(lst.count(2)) # 2의 갯수 
lst.clear() # 모든 요소 삭제
print(lst)

[1, 2, 2, 3]
[3, 2, 2, 1]
1
2
[]


# 3. set
- 중복 없이 요소를 저장하는, 순서 없는(hash 기반) 가변 컬렉션(mutable)
- hashable한 객체만 저장가능(mutable은 hashable하지 않음)

## 3.1 set의 생성방법

In [ ]:
# s = {}  # set이 아니라 dict
s = set() # 빈 set
s = {1, 2, 3}
s = set([1, 2, 2, 3]) # list -> set 중복제거
s = set("banana")   # {'b','a','n'} 중복은 저장안됨

## 3.2 set의 메서드
- add(x)       # x를 추가. mutable객체는 추가x
- remove(x)    # x가 없으면 KeyError
- discard(x)   # x를 찾아서 삭제(없으면 무시)
- pop()        # 임의 요소 꺼내서 반환(삭제)
- clear()      # 모든 요소 삭제
- union()         # 합집합. a | b
- intersection()  # 교집합. a & b
- difference()    # 차집합. a - b
- symmetric_difference() # XOR a ^ b
- issubset()    # 하위 집합인지 확인, >=
- issuperset()  # 상위 집합인지 확인,. <=
- isdisjoint()  # 교집합이 없는지 확인, !=

### hashable - 객체의 hash값이 변하지 않고 비교(==) 일관성을 갖는 것
- 객체가 __ hash__를 가져야하고, hash값이 불변이어야 한다.
- mutable객체는 변경될 수 있으므로 hash값이 비교 일관성을 가질 수 없다.

In [117]:
fs = frozenset([1,2,3]) # set(mutable) -> frozenset(immutable)
fs.add(4) # 에러. frozenset은 변경불가(immutable)

AttributeError: 'frozenset' object has no attribute 'add'

In [ ]:
s = {1,2,3}
for e in s: # set은 iterable. 순서보장x
    print(e)


1
2
3


# 4. dict
- (key, value)를 쌍으로 묶어서 저장하는 자료구조(hashtable)
- 순서 없음. key는 중복x, value는 중복o
- key는 반드시 hashable이어야 한다.

## 4.1 dict 생성방법

In [127]:
d = {}     # 빈 dict, set이 아님에 주의
d = dict() # 빈 dict
d = dict(a=1, b=2) # {'a': 1, 'b': 2}
dict([("a",1), ("b",2)]) # {'a': 1, 'b': 2}

keys = [*"abcd"]
values = [*range(4)]
d = dict(zip(keys, values)) # {'a': 0, 'b': 1, 'c': 2, 'd': 3}
print(d)

pairs = [('a', 1), ('b', 2)]
d = {k: v for k, v in pairs} # dict comprehension
print(d)


{'a': 0, 'b': 1, 'c': 2, 'd': 3}
{'a': 1, 'b': 2}


## 4.2 dict메서드

In [148]:
d = dict(a=1, b=2) # {'a': 1, 'b': 2}
print(d)
print(f"{len(d)=}")
print(d.get('a'))  # key 'a'와 연결된 값을 반환. 
d['a'] = 100 # 수정
print(d.get('a'))  # key 'a'와 연결된 값을 반환
d['c'] = 3 # ('c',3)추가
print(d)
del d['a'] # key 'a'를 삭제
print('a' in d) # False. d에 키'a'가 있는지 확인
print(d.get('a')) # key를 못찾으면 None반환
print(d.get('a', 0)) # key를 찾지 못하면 0을 반환


{'a': 1, 'b': 2}
len(d)=2
1
100
{'a': 100, 'b': 2, 'c': 3}
False
None
0


In [157]:
d = dict(a=1, b=2, c=3, d=4)
print(d)
for k in d.keys(): # key만 뽑아서 출력
    print(k, end=" ")
print()
for v in d.values(): # value만 뽑아서 출력
    print(v, end=" ")
print()
for v in d.items(): # (key, value)로 뽑아서 출력
    print(v, end=" ")


{'a': 1, 'b': 2, 'c': 3, 'd': 4}
a b c d 
1 2 3 4 
('a', 1) ('b', 2) ('c', 3) ('d', 4) 

In [161]:
from collections import Counter

c = Counter("do or don't. threre is no trying")
print(f"{c.get('o')=}")

c.get('o')=4


In [184]:
import copy

d = {'a': [1], 'b': [2]}
print(d)
d2 = d.copy()          # 앝은 복사
dc = copy.deepcopy(d)  # 깊은 복사
print("=== 변경전 ===")
print(f"{d2=}")
print(f"{dc=}")
d['a'][0] = 2
print("=== 변경후 ===")
print(f"{d2=}")
print(f"{dc=}")


{'a': [1], 'b': [2]}
=== 변경전 ===
d2={'a': [1], 'b': [2]}
dc={'a': [1], 'b': [2]}
=== 변경후 ===
d2={'a': [2], 'b': [2]}
dc={'a': [1], 'b': [2]}


## 4.3 OrderedDict - 순서를 유지
- python 3.7부터 dict가 순서를 유지. OrderDict사용하지 않아도 됨

## 4.4 namedtuple - 불변
- tuple의 확장. 자바의 record
- 튜플의 각 요소에 이름을 붙일 수 있다.

In [192]:
from collections import namedtuple

Point = namedtuple("Point", ["x", "y"])
p = Point(10, 20) # x=10, y=20
print(Point._fields)
print(f"{p.x=}")        # 10
print(f"{p[0]=}")        # 10
x, y = p   # 언패킹
print(f"{x=},{y=}")

('x', 'y')
p.x=10
p[0]=10
x=10,y=20


# 4.5 PriorityQueue
- 우선순위 큐. 내부적으로 heapq사용
- 멀티 쓰레드에 안전(thread-safe)
- blocking put() / get()

In [195]:
from queue import PriorityQueue

pq = PriorityQueue()   # 최소 힙(minimum heap)
pq.put((1, "task A"))
pq.put((2, "task C"))
pq.put((0, "task B"))
pq.get()   # (0, "task B")  # 최소값부터 꺼내짐


(0, 'task B')

### 주의할점
pq.put((1, obj))   
pq.put((1, obj2))  
- priority가 같으면
- obj < obj2 비교 시도
- 비교 불가 객체면 TypeError

### PriorityQueue vs. heapq
| 구분       | heapq        | PriorityQueue |
| -------- | ------------ | ------------- |
| 성격       | 알고리즘 도구      | 큐 추상화         |
| 타입       | 함수 모듈        | 클래스           |
| 내부 저장    | list         | heapq         |
| 스레드 안전   | ❌            | ⭕             |
| blocking | ❌            | ⭕             |
| 성능       | 매우 빠름        | 상대적으로 느림      |
| 유연성      | 매우 높음        | 제한적           |
| 사용 목적    | 단일 스레드, 알고리즘 | 멀티스레드 작업 큐    |


## 4.5 그 외의 자료구조
- collections.namedtuple : 이름붙은 튜플. 튜플의 확장
- 큐 : queue.Queue, LifoQueue, collections.deque
- 우선순위 큐 : PriorityQueue(thread-safe), heapq(최소힙)

## 4.6 컬렉션의 필수 메서드(정리)
| 분류              | 필수 메서드                         |
| --------------- | ------------------------------ |
| Iterable        | `__iter__`                     |
| Iterator        | `__iter__`, `__next__`         |
| Sequence        | `__len__`, `__getitem__`       |
| MutableSequence | `__setitem__`, `append`, `pop` |
| Set             | `__contains__`, 집합 연산          |
| Mapping         | `__getitem__`, `keys`          |



## 4.7 itertools

- 반복(iteration)을 효율적이고 깔끔하게 처리할 수 있도록 만들어진 도구 모음(표준 라이브러리)

In [196]:
import itertools as it


for i in dir(it):
    if not i.startswith("_"):
        print(i)

accumulate
batched
chain
combinations
combinations_with_replacement
compress
count
cycle
dropwhile
filterfalse
groupby
islice
pairwise
permutations
product
repeat
starmap
takewhile
tee
zip_longest


### 무한 반복자
- count(start, step) : 시작값에서 계속 증가하는 숫자 생성
- cycle(iterable) : 주어진 반복 가능한 객체를 무한히 순환
- repeat(elem, n) : 요소를 n번 또는 무한히 반복

In [202]:
from itertools import count, cycle, repeat

for i in count(10, 5): # 10부터 시작. 5씩 계속 증가
    print(i)  # 10, 15, 20, 25, ...
    if(i > 20) : break


10
15
20
25


### 조합 및 순열
- product(...) : 데카르트 곱(모든 조합 조합)
- permutations(...) : 순열(순서 고려, r 길이)
- combinations(...) : 조합(순서 없는 선택)

In [205]:
from itertools import product, permutations, combinations

list(product([1,2],[3,4]))  
# → [(1,3),(1,4),(2,3),(2,4)]

list(permutations([1,2,3], 2))
# → [(1,2),(1,3),(2,1),(2,3),(3,1),(3,2)]

list(combinations([1,2,3], 2))
# → [(1,2),(1,3),(2,3)]


[(1, 2), (1, 3), (2, 3)]

In [207]:
# 부분 집합 만들기
import itertools
def subset(n) :
    for i in range(0,n+1) :
        for i in itertools.combinations(range(0,n),i) :
            print(i)
            subset.s.append(set(i))

subset.s = []
subset(3)
subset.s

()
(0,)
(1,)
(2,)
(0, 1)
(0, 2)
(1, 2)
(0, 1, 2)


[set(), {0}, {1}, {2}, {0, 1}, {0, 2}, {1, 2}, {0, 1, 2}]

### 종료 반복자 (Terminating Iterators)
- 입력 데이터의 길이에 맞춰 결과를 반환합니다:
- chain(...) : 여러 반복 가능한 객체를 하나로 이어줌
- accumulate(...) : 누적 결과(합, 곱 등)
- islice(...) : 슬라이스처럼 특정 구간만 반복
- groupby(...) : 같은 키끼리 그룹화
- takewhile(...) / dropwhile(...) : 조건 만족 동안만/벗어날 때까

In [197]:
# list 요소의 값을 누적 [0,1,2,3,4] -> [0,1,3,6,10]
a = it.accumulate([x for x in range(5)])
list(a)

[0, 1, 3, 6, 10]

In [199]:
# 두 
gen = it.chain('ABC', 'DEF')
for i in gen:
    print(i, end=" ")

A B C D E F 